# 6. Motivation figures
**Purpose.** Produce W_1, W_2, and F1 (fit vs one-day-shifted forecast) for the 6 motivation.

**Inputs.** Yahoo Finance `^GSPC` and `^VIX` 1995-01-01 → 2022-05-15.

**Expected runtime.** ~2 min on M5 Pro CPU (Guyon TSPL fit dominates).

**Dependencies.** `code_section6.motivation`, `code_guyon.empirical_study.main_function`.

## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error

import code_section6.motivation as motiv
from empirical_study.main_function import perform_empirical_study


## Lag-0 weights $W_1, W_2$ (closed form, Table 3 params)

In [ ]:
TABLE3_VIX = dict(
    beta_0=0.057, beta_1=-0.095, beta_2=0.82,
    alpha_1=1.06, delta_1=0.020,
    alpha_2=1.60, delta_2=0.052,
)
W1 = motiv.lag_zero_weight(TABLE3_VIX['alpha_1'], TABLE3_VIX['delta_1'], 1/252)
W2 = motiv.lag_zero_weight(TABLE3_VIX['alpha_2'], TABLE3_VIX['delta_2'], 1/252)
print(f'W_1 = {100*W1:.3f}%   (R_1 lag-0 weight)')
print(f'W_2 = {100*W2:.3f}%   (R_2 lag-0 weight)')

## F1. fit vs one-day-shifted forecast

In [ ]:
spx, vix = motiv.fetch_spx_vix_yahoo('1995-01-01', '2022-05-15')
print('spx rows:', len(spx), '  vix rows:', len(vix))

In [ ]:
sol = perform_empirical_study(
    vol=vix, index=spx, p=1, tspl=True,
    setting=[(1, 1), (2, 1/2)],
    train_start_date=pd.Timestamp('2000-01-01'),
    test_start_date=pd.Timestamp('2019-01-01'),
    test_end_date=pd.Timestamp('2022-05-15'),
    max_delta=1000,
)
print({k: sol[k] for k in ['train_r2', 'test_r2', 'train_rmse', 'test_rmse']})

In [ ]:
p = sol['opt_params']
beta_0, beta_1, beta_2 = float(p['beta_0']), float(p['beta_1']), float(p['beta_2'])
alpha_1, alpha_2 = float(p['alpha_1']), float(p['alpha_2'])
delta_1, delta_2 = float(p['delta_1']), float(p['delta_2'])
W1_fit = M.lag_zero_weight(alpha_1, delta_1, 1/252)
W2_fit = M.lag_zero_weight(alpha_2, delta_2, 1/252)
print(f'fitted: beta_0={beta_0:.4f}  beta_1={beta_1:.4f}  beta_2={beta_2:.4f}')
print(f'        alpha_1={alpha_1:.3f}  delta_1={delta_1:.4f}  alpha_2={alpha_2:.3f}  delta_2={delta_2:.4f}')
print(f'W_1 (fit) = {100*W1_fit:.3f}%   W_2 (fit) = {100*W2_fit:.3f}%')

# One-day-ahead forecast available to a trader at end of day t-1: yesterday's
# same-day prediction. Display-shift only — same model, different time.
forecast_one_day = sol['test_pred'].shift(1)

In [ ]:
test_idx = sol['test_pred'].index
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test_idx, 100 * vix.reindex(test_idx),               label='VIX (true)',               lw=1.0)
ax.plot(test_idx, 100 * sol['test_pred'].reindex(test_idx),  label='Same-day fit',             lw=1.0)
ax.plot(test_idx, 100 * forecast_one_day.reindex(test_idx),  label='One-day-shifted forecast', lw=1.0)
ax.set_ylabel('VIX (%)')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'fit_vs_forecast_one_day_shift.pdf')
plt.show()

## Summary numbers 

In [ ]:


test_target = vix.reindex(test_idx).dropna()
same_day_rmse = root_mean_squared_error(test_target, sol['test_pred'].reindex(test_target.index))
fc_aligned = forecast_one_day.reindex(test_target.index).dropna()
shifted_rmse = root_mean_squared_error(test_target.reindex(fc_aligned.index), fc_aligned)
print(f'W_1 (Table 3) = {100*W1:.3f}%      W_2 (Table 3) = {100*W2:.3f}%')
print(f'W_1 (re-fit)  = {100*W1_fit:.3f}%      W_2 (re-fit)  = {100*W2_fit:.3f}%')
print(f'Same-day fit          test RMSE = {100*same_day_rmse:.3f}%')
print(f'One-day-shifted fcst  test RMSE = {100*shifted_rmse:.3f}%')
print(f'Degradation factor              = {shifted_rmse / same_day_rmse:.2f}x')